# Notebook 06: Capstone Demonstration (4 Core Scenarios & Human Review)

Demonstrating all four project scenarios:
1. **Normal Case**: Baseline steady-state milling
2. **Edge Case**: Borderline thermal drift requiring monitoring
3. **Failure Case**: Hardware sensor open-circuit fault
4. **High-Risk Case**: Severe dual-parameter excursion with live Human Escalation and Sign-Off


In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from backend.app.agent.graph import ManufacturingAgentExecutor
from backend.app.agent.review import get_review_manager

executor = ManufacturingAgentExecutor()
manager = get_review_manager()

def print_scenario_result(title, state):
    print('=' * 75)
    print(f'SCENARIO: {title}')
    print('=' * 75)
    print(f"INPUT: Machine {state['machine_id']} | Query: {state['user_query']}")
    print(f"RISK LEVEL: {state['risk_level']} (Reason: {state['risk_reason']})")
    print(f"TOOLS INVOKED: {state['selected_tools']}")
    print(f"EVIDENCE COUNT: {len(state['evidence'])}")
    print(f"REVIEW STATUS: {state['review_status']}")
    print('-' * 75)
    print(f"FINAL RESULT:\n{state['final_response'][:400]}...\n")


## 1. Normal Case Demonstration


In [2]:
res_normal = executor.run(
    machine_id='M-101',
    user_query='Check if current telemetry conforms to ISO 10816 standards.',
    sensor_data={'temperature': 52.4, 'vibration': 0.85, 'pressure': 5.4, 'speed': 4500, 'status': 'RUNNING'}
)
print_scenario_result('Case 1: Normal Operation', res_normal)


SCENARIO: Case 1: Normal Operation
INPUT: Machine M-101 | Query: Check if current telemetry conforms to ISO 10816 standards.
RISK LEVEL: NORMAL (Reason: All measured telemetry parameters are strictly within nominal manufacturer tolerances.)
TOOLS INVOKED: ['retrieve_manufacturing_guidelines']
EVIDENCE COUNT: 3
REVIEW STATUS: approved
---------------------------------------------------------------------------
FINAL RESULT:
Review Status: human_review
Quality Score: 0.96
Audit Rationale: Telemetry violates critical ISO 10816 / thermal limits. High-risk operational threshold detected. Mandatory human escalation triggered before any maintenance dispatch.
Mandatory Human Review: True...



## 2. Edge Case Demonstration


In [3]:
res_edge = executor.run(
    machine_id='M-102',
    user_query='Spindle temperature reached 74.8C. Advise on inspection.',
    sensor_data={'temperature': 74.8, 'vibration': 2.65, 'pressure': 4.1, 'speed': 6200, 'status': 'WARN_TELEMETRY'}
)
print_scenario_result('Case 2: Edge Condition', res_edge)


SCENARIO: Case 2: Edge Condition
INPUT: Machine M-102 | Query: Spindle temperature reached 74.8C. Advise on inspection.
RISK LEVEL: EDGE (Reason: Spindle temperature 74.8°C in advisory drift zone (68.0°C - 82.0°C). | Vibration velocity 2.65 mm/s RMS in Class B warning zone (1.8 - 3.8 mm/s). | Hydraulic line pressure 4.1 bar in warning band (nominal 4.5-6.5 bar). | Multiple concurrent telemetry warnings detected; close monitoring advised.)
TOOLS INVOKED: ['retrieve_manufacturing_guidelines', 'get_machine_history', 'calculate']
EVIDENCE COUNT: 6
REVIEW STATUS: approved
---------------------------------------------------------------------------
FINAL RESULT:
### 1. Telemetry & Risk Summary
- Evaluated telemetry indicates asset operating profile is actively monitored.
- Key parameters: Temperature, Vibration RMS, Pressure, and Spindle Speed evaluated against technical thresholds.

### 2. Grounded Technical Evidence (From Verified SOPs & Manuals)
- **machine_manual.md (Section 2)**: Nominal

## 3. Failure Case Demonstration


In [4]:
res_fail = executor.run(
    machine_id='M-301',
    user_query='Sensor reporting negative numbers.',
    sensor_data={'temperature': -999.0, 'vibration': 0.0, 'pressure': -1.0, 'speed': 0, 'status': 'SENSOR_FAULT'}
)
print_scenario_result('Case 3: Sensor Fault / Non-physical Telemetry', res_fail)


SCENARIO: Case 3: Sensor Fault / Non-physical Telemetry
INPUT: Machine M-301 | Query: Sensor reporting negative numbers.
RISK LEVEL: HIGH (Reason: Temperature sensor reporting non-physical reading (-999.0°C). Potential transducer failure. | Hydraulic pressure sensor reporting non-physical reading (-1.0 bar). | Multiple critical sensor deviations detected; compound operational failure risk.)
TOOLS INVOKED: ['retrieve_manufacturing_guidelines', 'get_machine_history', 'calculate', 'request_human_review']
EVIDENCE COUNT: 6
REVIEW STATUS: human_review
---------------------------------------------------------------------------
FINAL RESULT:
### 1. Telemetry & Risk Summary
- Evaluated telemetry indicates asset operating profile is actively monitored.
- Key parameters: Temperature, Vibration RMS, Pressure, and Spindle Speed evaluated against technical thresholds.

### 2. Grounded Technical Evidence (From Verified SOPs & Manuals)
- **machine_manual.md (Section 2)**: Nominal spindle temperature 

## 4. High-Risk Case Demonstration & Human Review Sign-off


In [5]:
res_high = executor.run(
    machine_id='M-201',
    user_query='Extreme chatter, 89.6C heat and 5.42 mm/s vibration reported during heavy titanium turn.',
    sensor_data={'temperature': 89.6, 'vibration': 5.42, 'pressure': 2.9, 'speed': 8200, 'status': 'ELEVATED_RISK'}
)
print_scenario_result('Case 4: High Risk Escalation', res_high)

# Inspect Pending Ticket
pending = manager.list_pending_tickets()
print(f'Pending Tickets in Human Review Queue: {len(pending)}')
ticket_id = res_high['request_id']
print(f'Reviewing Ticket ID: {ticket_id}')

# Human Engineer Sign-off Action
signoff = manager.submit_decision(
    request_id=ticket_id,
    decision='approve',
    notes='Confirmed severe bearing raceway spalling. Dispatched mechanical overhaul technician with replacement bearing pack.',
    reviewer_id='CHIEF_MAINT_ENG_402'
)
print(f'Human Decision Recorded: {signoff.decision.upper()} by {signoff.reviewer_id}')
print(f'Notes: {signoff.notes}')


SCENARIO: Case 4: High Risk Escalation
INPUT: Machine M-201 | Query: Extreme chatter, 89.6C heat and 5.42 mm/s vibration reported during heavy titanium turn.
RISK LEVEL: HIGH (Reason: Spindle temperature 89.6°C breaches critical limit (> 82.0°C). High risk of bearing seizure. | Vibration velocity 5.42 mm/s RMS exceeds ISO 10816 critical threshold (> 3.8 mm/s). High risk of tool breakage or bearing spalling. | Hydraulic line pressure 2.9 bar critically low (< 3.8 bar). Risk of tool unclamping in cut. | Multiple critical sensor deviations detected; compound operational failure risk.)
TOOLS INVOKED: ['retrieve_manufacturing_guidelines', 'get_machine_history', 'calculate', 'request_human_review']
EVIDENCE COUNT: 6
REVIEW STATUS: human_review
---------------------------------------------------------------------------
FINAL RESULT:
### 1. Telemetry & Risk Summary
- Evaluated telemetry indicates asset operating profile is actively monitored.
- Key parameters: Temperature, Vibration RMS, Press